## 1. Find the files


In [10]:
from pathlib import Path
import pandas as pd
pd.set_option("display.max_colwidth", None)

RAW = Path("../data/raw")
csv_path = next(RAW.glob("Most-Recent-Cohorts-Institution*.csv"))
dict_path = next(RAW.glob("*Dictionary*.xlsx"))
print(csv_path.name, "|", dict_path.name)

           

Most-Recent-Cohorts-Institution.csv | ~$CollegeScorecardDataDictionary.xlsx


## 2. Count privacy suppressed ("PS") values before cleaning 
PS meaning: Earnings, debt and loan data that did not meet reporting standards.


In [5]:
raw = pd.read_csv(csv_path, low_memory=False)
ps_counts = (raw == "PS").sum().sort_values(ascending = False)
print(f"{(ps_counts > 0).sum()} columns contain 'PS' values")
ps_counts.head(10)


2476 columns contain 'PS' values


BBRR4_PP_DEP_PAIDINFULL    6100
BBRR4_PP_DEP_NOPROG        6100
BBRR4_PP_IND_DFLT          6100
BBRR1_PP_IND_N             6100
BBRR4_PP_IND_DLNQ          6100
BBRR4_PP_DEP_DISCHARGE     6100
BBRR4_PP_IND_FBR           6100
BBRR4_PP_DEP_MAKEPROG      6100
BBRR4_PP_DEP_DFR           6100
BBRR4_PP_IND_MAKEPROG      6100
dtype: int64

## 3. Reload with "PS" treated as missing

In [6]:
df = pd.read_csv(csv_path, na_values=["PS"], low_memory=False)
del raw
print(df.shape)
print(df.dtypes.value_counts())
df[["UNITID", "INSTNM", "STABBR", "CONTROL"]].head()
            

(6272, 3379)
float64    2672
str         693
int64        14
Name: count, dtype: int64


,UNITID,INSTNM,STABBR,CONTROL
0,100654,Alabama A & M University,AL,1
1,100663,University of Alabama at Birmingham,AL,1
2,100690,Amridge University,AL,2
3,100706,University of Alabama in Huntsville,AL,1
4,100724,Alabama State University,AL,1


## 4. List what is in the data dictionary

In [7]:
sheets = pd.read_excel(dict_path, sheet_name=None)
for name, sheet in sheets.items():
    print(name, sheet.shape)

README (26, 1)
ChangeLog (144, 3)
Glossary (16, 3)
Institution_Data_Dictionary (3736, 11)
Institution_Cohort_Map (3379, 31)
Most_Recent_Inst_Cohort_Map (3379, 3)
FieldOfStudy_Data_Dictionary (244, 10)
FieldOfStudy_Cohort_Map (230, 11)


### 4a. Variable lookup 


In [11]:
def lookup(var):
    for name, sheet in sheets.items():
        match = sheet.astype(str).apply(lambda col: col.str.contains(var, case=False, regex=False))
        hits = sheet[match.any(axis=1)]
        if not hits.empty:
            print(f"--- sheet: {name}")
            display(hits.dropna(axis=1, how="all"))

lookup("BBRR4_PP_DEP_PAIDINFULL")

            

--- sheet: Institution_Data_Dictionary


,NAME OF DATA ELEMENT,dev-category,developer-friendly name,API data type,VARIABLE NAME,SOURCE
3440,Percentage of dependent student Parent PLUS Loan borrowers paid in full after 4 years,repayment,4_yr_bb_pp_repayment.dependent.fullypaid,string,BBRR4_PP_DEP_PAIDINFULL,NSLDS


--- sheet: Institution_Cohort_Map


,variable,Merged2020_21
3176,BBRR4_PP_DEP_PAIDINFULL,"NSLDS AY2015-16, AY2016-17 pooled cohorts measured in 2019-2020, 2020-2021"


--- sheet: Most_Recent_Inst_Cohort_Map


,variable,"Cohort/Measure in ""Most Recent"" Datafile",Merged file Drawn From (MergedXXXX_YY_PP.csv)
3176,BBRR4_PP_DEP_PAIDINFULL,"NSLDS AY2015-16, AY2016-17 pooled cohorts measured in 2019-2020, 2020-2021",2020_21
